# Topic 8: Quantization, Pruning, and Knowledge Distillation

**Day 3 of 3 -- Final Topic**

In Topic 7b you assembled the Barclays Complaint Intelligence System using PEFT/LoRA on DistilBERT.
That model is accurate. Now you will make it production-ready: smaller, faster, and cheap to serve.

This notebook picks up from where Topic 7b left off. The PEFT checkpoint you saved in T7b is the
starting point for the compression pipeline you build here.

**Note**: Each section loads the checkpoint fresh from S3 -- you do not need to re-run T7b.
The kernel state is independent, but the logical progression is continuous.

### What You Will Build

By the end of this notebook you will have:

1. Applied Post-Training Quantization (PTQ) -- int8 weights, no retraining
2. Run Quantization-Aware Training (QAT) -- simulate quantization during fine-tuning
3. Applied structured pruning -- zero out low-importance attention heads
4. Run knowledge distillation -- transfer T7b DistilBERT knowledge to a smaller student
5. Deployed the compressed model to a SageMaker endpoint
6. Replaced the Topic 1 GPT-4o API call with your own endpoint call

### Prerequisites

- Completed Topic 7b or have a DistilBERT PEFT checkpoint in S3
- SageMaker execution role with S3 and ECR access
- `ml.g4dn.xlarge` instance available in your AWS account


In [ ]:
# Disable TensorFlow backend in transformers (SageMaker image compatibility).
# Must run before any transformers import.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


In [ ]:
# Install pinned dependencies
# numpy<2 is mandatory because numpy 2.x breaks many torch operations.
!pip install -q \
    "sagemaker>=2.200.0,<3.0.0" \
    "transformers>=4.53,<4.54" \
    "accelerate>=1.0.0" \
    "tokenizers>=0.21,<0.22" \
    "datasets>=2.18.0,<3.0.0" \
    "peft>=0.6.0" \
    "numpy<2"

import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.ao.quantization
import torch.nn.functional as F
import numpy as np
import sagemaker
from sagemaker import get_execution_role
from sagemaker.huggingface import HuggingFace
import boto3

# Reproducibility helper used across the course
def set_seeds(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seeds(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SageMaker session uses the execution role attached to this Studio instance
sess   = sagemaker.Session()
role   = get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name

print(f"Role:    {role}")
print(f"Bucket:  {bucket}")
print(f"Region:  {region}")
print(f"Device:  {device}")
print(f"PyTorch: {torch.__version__}")


## CUDA Health Check

This kernel may be running on a GPU instance or a CPU one depending on quota.
The cell below checks whether a GPU is present and whether `torch` can use it.
If a GPU is present but `torch` is a CPU-only build, it reinstalls a matching
CUDA wheel. If you see a reinstall message, restart the kernel before continuing.


In [ ]:
# CUDA health check + safety-net.
# A SageMaker kernel may land on a GPU instance (ml.g4dn.xlarge) or a CPU one
# (ml.t3.medium) depending on quota. On a GPU box the image's torch is sometimes
# CPU-only, so torch.cuda.is_available() is wrongly False. This cell detects the
# real situation from two signals (nvidia-smi + the SageMaker metadata file) and
# reinstalls a matching CUDA wheel only when needed.
import json
import os
import re
import shutil
import subprocess
import sys

import torch


def _sagemaker_image():
    # SageMaker writes the running image/instance here. Diagnostic only.
    path = "/opt/ml/metadata/resource-metadata.json"
    if not os.path.exists(path):
        return None
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def _nvidia_smi():
    # Returns (gpu_present, driver_cuda_version_str_or_None).
    # nvidia-smi exit 0 means GPU hardware is here even if torch cannot see it.
    if shutil.which("nvidia-smi") is None:
        return False, None
    try:
        out = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=15)
    except Exception:
        return False, None
    if out.returncode != 0:
        return False, None
    m = re.search(r"CUDA Version:\s*([0-9]+\.[0-9]+)", out.stdout)
    return True, (m.group(1) if m else None)


def _pick_wheel(driver_cuda):
    # Newest cuXXX wheel index <= the driver's max CUDA.
    # SageMaker g4dn (T4) images historically ship CUDA 12.1-12.4 drivers.
    tiers = [("12.6", "cu126"), ("12.4", "cu124"), ("12.1", "cu121")]
    if not driver_cuda:
        return "cu121"  # safe default when the driver version is unreadable
    try:
        dv = tuple(int(x) for x in driver_cuda.split("."))
    except ValueError:
        return "cu121"
    for ver, tag in tiers:
        if tuple(int(x) for x in ver.split(".")) <= dv:
            return tag
    return "cu121"


meta = _sagemaker_image()
if meta:
    print("SageMaker kernel:", meta.get("AppType", "?"),
          "| instance:", meta.get("ResourceArn", "?").split("/")[-1])

has_gpu, driver_cuda = _nvidia_smi()

if not has_gpu:
    print("No GPU detected. Running on CPU kernel. No action needed.")
elif torch.version.cuda is not None:
    print(f"GPU OK. torch {torch.__version__} built for CUDA {torch.version.cuda}.")
    print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
else:
    wheel = _pick_wheel(driver_cuda)
    print(f"GPU present (driver CUDA {driver_cuda}) but torch is CPU-only.")
    print(f"Reinstalling torch from the {wheel} wheel index...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade",
         "torch", "torchvision", "torchaudio",
         "--index-url", f"https://download.pytorch.org/whl/{wheel}"],
        check=True,
    )
    print("Reinstalled torch with CUDA support.")
    print("RESTART THE KERNEL now, then re-run the notebook from the top.")


> **Note**: Each section loads checkpoints from S3. Kernel state is fresh, but logically this continues from T7b.

## Day 3 System Overview -- Where You Are in the Journey

This is the FINAL topic of the course. By the time you finish Topic 8, the Barclays
complaint-intelligence assistant is production-ready and cost-efficient.

| Day | Topics | What You Built |
|-----|--------|---------------|
| Day 1 | T1-T4 | GPT-4o prototype --> attention --> full Transformer from scratch |
| Day 2 | T5-T7b | HuggingFace ecosystem --> fine-tuning --> PEFT/LoRA assembly |
| **Day 3** | **T8 (this notebook)** | **Quantization + pruning + distillation --> deployable endpoint** |

### YOU ARE HERE: Day 3 -- Topic 8 (1 of 1)

The fine-tuned, LoRA-adapted DistilBERT you shipped in T7b is accurate but heavy.
Topic 8 teaches you to make it fast and cheap without sacrificing quality:

- **Post-Training Quantization (PTQ)**: shrink weights from float32 to int8 with no retraining
- **Quantization-Aware Training (QAT)**: simulate quantization during fine-tuning for better accuracy
- **Structured Pruning**: zero out entire attention heads to cut inference FLOPs
- **Knowledge Distillation**: transfer T7b DistilBERT knowledge into a smaller student model
- **SageMaker Endpoint**: deploy the compressed model as a real REST endpoint

The T1 demo called GPT-4o with a raw complaint. By the end of this notebook you will have
replaced that API call with your own fine-tuned, quantized, pruned, distilled model --
served from a SageMaker endpoint your code controls.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import time
import os

# Load a fresh DistilBERT classifier. Note: this is NOT carried over from Topic 7b
# kernel state. We re-load distilbert-base-uncased here so Topic 8 is self-contained
# and students who lost their T7b kernel can still run all compression demos.
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5,
)
baseline_model.eval()

# Measure model size in FP32
param_count = sum(p.numel() for p in baseline_model.parameters())
param_size_mb = param_count * 4 / (1024 ** 2)
print(f"Parameter count   : {param_count:,}")
print(f"Model size (FP32) : {param_size_mb:.1f} MB")

# Measure inference latency on CPU
sample_text = "I was charged twice for the same transaction and nobody has responded to my complaint."
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=128)

with torch.no_grad():
    _ = baseline_model(**inputs)

latencies = []
for _ in range(20):
    start = time.perf_counter()
    with torch.no_grad():
        _ = baseline_model(**inputs)
    latencies.append((time.perf_counter() - start) * 1000)

avg_ms = float(np.mean(latencies))
print(f"Avg CPU inference : {avg_ms:.1f} ms")
print(f"Endpoint cost est : $0.074/hr (ml.m5.xlarge) for a dedicated endpoint")
print()
print("Production target: <80 MB, <20 ms, <$0.03/hr equivalent throughput")


## Section 1: Quantization

Weights are stored as FP32 by default: 32 bits per number, 4 bytes per weight. Quantization maps those floats to a smaller type. INT8 uses 8 bits (1 byte), giving a 4x size reduction.

The key question: how much accuracy do we sacrifice?

Two strategies:

- Post-Training Quantization (PTQ): quantize AFTER training. Simple, but can lose 2 to 5 percent accuracy.
- Quantization-Aware Training (QAT): simulate quantization DURING training. Recovers up to 96 percent of the accuracy degraded by PTQ (PyTorch 2025 benchmarks on Llama3).

We will see both. Beat 1 shows PTQ going wrong. Beat 3 shows PTQ done right. The capstone runs full QAT.


### Beat 1: The PTQ Trap. What Happens When We Quantize Without Calibration

PTQ requires a calibration step: you run a representative sample of data through the model to collect activation statistics (min, max). Without calibration, the quantized ranges are estimated from weights alone and activation distributions are ignored.

Watch what happens when we skip calibration and quantize naively.


In [ ]:
# NAIVE PTQ: quantize the model with no calibration data at all.
# torch.ao.quantization.quantize_dynamic is the simplest PTQ path.
# It quantizes weights only (not activations). Watch what happens to the output distribution.

naive_ptq_model = torch.ao.quantization.quantize_dynamic(
    baseline_model,
    {nn.Linear},
    dtype=torch.qint8,
)

complaint_texts = [
    "My card was declined three times at the ATM and I lost money.",
    "I never received the international wire transfer I sent last week.",
    "I need to update my address on the account.",
]

print("=== Logit distributions: FP32 baseline vs naive PTQ ===")
print()
for text in complaint_texts:
    inputs_local = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        fp32_logits = baseline_model(**inputs_local).logits[0].numpy()
        ptq_logits  = naive_ptq_model(**inputs_local).logits[0].numpy()
    fp32_pred = int(np.argmax(fp32_logits))
    ptq_pred  = int(np.argmax(ptq_logits))
    print(f"Text    : {text[:60]}...")
    print(f"FP32    : pred={fp32_pred}, logits={np.round(fp32_logits, 2)}")
    print(f"NaivePTQ: pred={ptq_pred}, logits={np.round(ptq_logits, 2)}")
    if fp32_pred != ptq_pred:
        print("  PREDICTION CHANGED: quantization error")
    print()

# Size estimate: quantized linear weights are int8 (1 byte), other params remain FP32 (4 bytes)
ptq_size_mb = sum(
    p.numel() * (1 if p.dtype == torch.qint8 else 4)
    for p in naive_ptq_model.parameters()
) / (1024 ** 2)
print(f"Naive PTQ model size estimate: {ptq_size_mb:.1f} MB (vs {param_size_mb:.1f} MB FP32)")
print()
print("Problem: we got a smaller model, but logit magnitudes shifted noticeably.")
print("With no calibration data, quantization ranges are estimated from weights alone.")
print("Activation distributions are ignored, which can cause errors on real inputs.")


### Beat 2: Understanding the Precision Tradeoffs

The diagram below shows how reducing bit-width reduces memory 4x to 8x, but also compresses the representable range of values, which is exactly what damages accuracy when no calibration is done.

QAT inserts fake-quantization operators during training that simulate this compression, so the model learns to be robust to the rounding errors before they happen for real.

<!-- DIAGRAM: Quantization precision tradeoffs -->
[View diagram](../../plans/topic_8_quantization/diagrams/quantization-precision-tradeoffs.mmd)


In [ ]:
# PROPER PTQ: static quantization with calibration.
# Steps:
#   1. Set qconfig (tells PyTorch what observers to insert)
#   2. torch.ao.quantization.prepare(): inserts observers
#   3. Run calibration data through model (observers collect statistics)
#   4. torch.ao.quantization.convert(): replaces observers with quantized ops

from datasets import load_dataset
import copy

# Load a small calibration set (100 examples from banking77)
calib_dataset = load_dataset("PolyAI/banking77", trust_remote_code=True, split="test[:100]")

# Clone the baseline model for PTQ so we do not modify the original
ptq_model = copy.deepcopy(baseline_model)
ptq_model.eval()

# Step 1: set quantization config (fbgemm targets x86 CPUs, used by SageMaker endpoints)
ptq_model.qconfig = torch.ao.quantization.get_default_qconfig("fbgemm")

# Embedding and LayerNorm layers cannot be quantized by torch.ao, opt them out
for name, module in ptq_model.named_modules():
    if isinstance(module, (nn.Embedding, nn.LayerNorm)):
        module.qconfig = None

# Step 2: prepare (inserts calibration observers)
torch.ao.quantization.prepare(ptq_model, inplace=True)

# Step 3: calibration. Run real data through the model
print("Running calibration (100 examples)...")
with torch.no_grad():
    for i, example in enumerate(calib_dataset):
        inputs_c = tokenizer(
            example["text"],
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding="max_length",
        )
        ptq_model(**inputs_c)
        if (i + 1) % 25 == 0:
            print(f"  Calibrated {i + 1}/100 examples")

# Step 4: convert. Replace observers with real INT8 quantized ops
ptq_model = torch.ao.quantization.convert(ptq_model, inplace=False)
print("PTQ with calibration complete.")
print()

print("=== After calibrated PTQ: predictions vs FP32 ===")
for text in complaint_texts:
    inputs_local = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        fp32_logits = baseline_model(**inputs_local).logits[0].numpy()
        ptq_logits  = ptq_model(**inputs_local).logits[0].numpy()
    print(f"FP32 pred={int(np.argmax(fp32_logits))} | PTQ pred={int(np.argmax(ptq_logits))} | {text[:55]}...")
print()
print("With calibration, PTQ predictions are stable. But QAT (capstone) recovers")
print("even more accuracy because the model saw quantized ops during training.")


### Lab 1: Dynamic Quantization on Your Complaint Classifier (Tier 1 Guided, 15 min)

**Situation**: You are the ML engineer at Barclays. You have a trained DistilBERT complaint classifier and need to produce a quick INT8 version for the on-premise inference server (which is x86 CPU only).

**Task**: Apply dynamic quantization to the model and measure the size and latency improvement.

**Action**: Follow the steps below. Each `= None  # YOUR CODE` line is one step.

**Result**: You should see at least 2x size reduction and a measurable latency improvement.

Steps:

1. Apply `torch.ao.quantization.quantize_dynamic` to `baseline_model`, targeting `{nn.Linear}`, with `dtype=torch.qint8`. Save as `dynamic_quantized_model`.
2. Run inference on `inputs` with `dynamic_quantized_model` inside the timed loop.
3. Compute the speedup ratio: `avg_ms / avg_ms_dynamic`.

Stretch: can you apply the same dynamic quantization only to the encoder layers (not the classifier head)? Hint: use `baseline_model.distilbert` as the target module instead of the full model.


In [ ]:
# ============================================================
# Lab 1: Dynamic Quantization on the Complaint Classifier
# ============================================================
import tempfile, os

# Step 1: apply dynamic quantization
# Hint: torch.ao.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
dynamic_quantized_model = None  # YOUR CODE

# Step 2: measure file size by saving to disk
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "model.pt")
    torch.save(dynamic_quantized_model, path)
    dq_size_mb = os.path.getsize(path) / (1024 ** 2)
print(f"Dynamic quantized model size: {dq_size_mb:.1f} MB  (baseline: {param_size_mb:.1f} MB)")

# Step 3: timed inference
dynamic_latencies = []
for _ in range(20):
    start = time.perf_counter()
    with torch.no_grad():
        # Hint: run inference with dynamic_quantized_model on inputs
        _ = None  # YOUR CODE
    dynamic_latencies.append((time.perf_counter() - start) * 1000)
avg_ms_dynamic = float(np.mean(dynamic_latencies))
print(f"Dynamic quantized avg latency: {avg_ms_dynamic:.1f} ms  (baseline: {avg_ms:.1f} ms)")

# Step 4: speedup ratio
# Hint: speedup is baseline latency divided by quantized latency
speedup = None  # YOUR CODE
print(f"Speedup: {speedup:.2f}x")

# Verification
assert dynamic_quantized_model is not None, "Apply quantize_dynamic first."
assert dq_size_mb < param_size_mb * 0.8, "Expected at least 20% size reduction."
print("Lab 1 complete.")


In [ ]:
# Safety-net: run this if you did not finish Lab 1.
# SKIP this cell if you DID finish Lab 1.
if dynamic_quantized_model is None:
    print("Using Lab 1 safety-net so the rest of the notebook can run.")
    dynamic_quantized_model = torch.ao.quantization.quantize_dynamic(
        baseline_model, {nn.Linear}, dtype=torch.qint8
    )
    import tempfile, os
    with tempfile.TemporaryDirectory() as tmpdir:
        path = os.path.join(tmpdir, "model.pt")
        torch.save(dynamic_quantized_model, path)
        dq_size_mb = os.path.getsize(path) / (1024 ** 2)
    avg_ms_dynamic = avg_ms * 0.7
    speedup = avg_ms / avg_ms_dynamic
    print(f"Safety-net: dynamic quantized model {dq_size_mb:.1f} MB, speedup {speedup:.2f}x")


#### Homework Extension: INT4 Quantization with bitsandbytes

Dynamic quantization gives INT8 precision. For even more aggressive compression, the `bitsandbytes` library supports INT4 (NF4) quantization, which is what QLoRA uses.

After class, try this:


In [ ]:
# Homework Extension: INT4 with bitsandbytes (run after class)
# Requires a CUDA GPU. Skip on CPU-only notebooks.
#
# from transformers import BitsAndBytesConfig, AutoModelForSequenceClassification
# bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
# model_4bit = AutoModelForSequenceClassification.from_pretrained(
#     "distilbert-base-uncased",
#     quantization_config=bnb_config,
#     num_labels=5,
# )
# Question to explore: can you fine-tune a 4-bit model with LoRA? What accuracy tradeoff do you see?
print("Homework: try INT4 with bitsandbytes after class. See commented code above.")


**Peer Discussion (3 min)**

We just saw dynamic quantization cut model size by around 4x with minimal accuracy loss.

1. Barclays runs 50 NLP models in production. Which ones would you quantize first and why?
2. Dynamic quantization works at inference time. What are the risks of quantizing a model that was fine-tuned on imbalanced complaint data?
3. INT8 loses some precision. For a fraud detection model, is a 0.5 percent accuracy drop acceptable? How would you decide?


## Section 2: Weight Pruning

Quantization reduces precision. Pruning takes a different approach: it zeros out weights entirely. A weight of exactly 0 contributes nothing to the output. If we zero 30 percent of weights, we need 30 percent fewer multiplications (with the right sparse runtime).

Two pruning strategies:

- Unstructured: zero individual weights anywhere in the weight matrix. Maximum flexibility, but requires sparse matrix kernels to realize speedup. Default in PyTorch's prune module.
- Structured: zero entire rows, columns, or attention heads. Reduces the actual matrix shape, so standard dense kernels get faster automatically. Harder to implement but immediately faster.

Beat 1: prune too aggressively (80 percent) and watch the model collapse.
Beat 3: prune conservatively (20 percent) with magnitude-based L1 and verify accuracy holds.


In [ ]:
# NAIVE PRUNING: remove 80% of weights globally. Too aggressive.
import copy

aggressive_model = copy.deepcopy(baseline_model)

# Apply L1 unstructured pruning: zero out the 80% of weights with smallest magnitude
for name, module in aggressive_model.named_modules():
    if isinstance(module, nn.Linear):
        prune.l1_unstructured(module, name="weight", amount=0.80)
        prune.remove(module, "weight")  # make pruning permanent

aggressive_model.eval()

print("=== Predictions after 80% aggressive pruning ===")
label_names = ["Card/Account", "Transaction Dispute", "FX/International", "General Query", "Other"]
for text in complaint_texts:
    inputs_local = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        logits = aggressive_model(**inputs_local).logits[0].numpy()
    pred = int(np.argmax(logits))
    confidence = float(torch.softmax(torch.tensor(logits), dim=0)[pred])
    print(f"Pred: {label_names[pred]} (conf={confidence:.2%}) | {text[:50]}...")

print()
print("Problem: 80% pruning collapses the model. Predictions may cluster on one class.")
print("The model has lost so much capacity it defaults to the most common category.")
print()
nonzero = sum((p != 0).sum().item() for p in aggressive_model.parameters())
total   = sum(p.numel() for p in aggressive_model.parameters())
print(f"Non-zero weights: {nonzero:,} of {total:,} ({100*nonzero/total:.1f}%)")


#### Discussion (3 min): How Much Pruning Is Safe?

Consider the 80 percent pruning result you just saw. Now consider your production context at Barclays:

- The complaint triage system routes 10,000 complaints per day to the right team. If 5 percent of complaints are mis-routed, that means 500 customers per day get the wrong response.
- Your infrastructure team is asking for a 3x compute reduction to cut costs.
- You have 2 hours of GPU time remaining for retraining.

Questions:

1. What pruning ratio would you propose, and how would you test it before deployment?
2. If pruning alone cannot achieve 3x speedup safely, what would you combine it with?
3. Should the pruning decision be made by the ML team, the product team, or both?


In [ ]:
# CONSERVATIVE PRUNING: 20% L1 unstructured. Safe for DistilBERT.
import copy

pruned_model = copy.deepcopy(baseline_model)

def sparsity(model):
    total   = sum(p.numel() for p in model.parameters())
    nonzero = sum((p != 0).sum().item() for p in model.parameters())
    return 100.0 * (1 - nonzero / total)

print(f"Sparsity before pruning: {sparsity(pruned_model):.2f}%")

# Apply L1 unstructured pruning to all Linear layers.
# L1 unstructured zeroes the weights with the smallest absolute value.
# amount=0.20 means 20% of weights in each Linear layer will be zeroed.
for name, module in pruned_model.named_modules():
    if isinstance(module, nn.Linear):
        prune.l1_unstructured(module, name="weight", amount=0.20)
        prune.remove(module, "weight")  # bake in the mask

print(f"Sparsity after  pruning: {sparsity(pruned_model):.2f}%")

pruned_model.eval()
print()
print("=== Predictions after 20% L1 pruning ===")
for text in complaint_texts:
    inputs_local = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        fp32_logits   = baseline_model(**inputs_local).logits[0].numpy()
        pruned_logits = pruned_model(**inputs_local).logits[0].numpy()
    fp32_pred   = int(np.argmax(fp32_logits))
    pruned_pred = int(np.argmax(pruned_logits))
    match = "OK" if fp32_pred == pruned_pred else "CHANGED"
    print(f"[{match}] FP32={fp32_pred} Pruned={pruned_pred} | {text[:55]}...")

print()
print("Note: PyTorch dense kernels do not skip zero-weights.")
print("Real speedup from unstructured pruning requires torch.sparse or hardware sparse support.")
print("Structured pruning (next lab) gives immediate speedup with standard dense kernels.")


### Lab 2: Global Magnitude Pruning (Tier 1 Guided, 15 min)

**Situation**: The Barclays infrastructure team says dense sparse ops will not be available on the production hardware for 6 months. You need pruning that delivers real speedup later, but right now you need to know the safest global rate to apply.

**Task**: Implement global magnitude-based pruning that zeroes 30 percent of weights across ALL Linear layers simultaneously (not 30 percent per layer), then measure the impact on model predictions.

**Action**: Fill in each `= None  # YOUR CODE` block.

**Result**: Print the global sparsity percentage and verify that predictions are stable.

Steps:

1. Use `prune.global_unstructured` with `parameters_to_prune`, `pruning_method=prune.L1Unstructured`, and `amount=0.30`.
2. Call `prune.remove(module, "weight")` on each Linear layer to make pruning permanent.
3. Compute and print the global sparsity of `global_pruned_model`.
4. Run the three complaint texts through `global_pruned_model` and compare predictions to FP32.

Stretch: try `amount=0.50`. At what point do predictions start changing for the complaint texts?


In [ ]:
# ============================================================
# Lab 2: Global Unstructured Pruning
# ============================================================
import copy

global_pruned_model = copy.deepcopy(baseline_model)

# Collect (module, parameter_name) tuples for all Linear layers
parameters_to_prune = [
    (module, "weight")
    for name, module in global_pruned_model.named_modules()
    if isinstance(module, nn.Linear)
]

# Step 1: apply global L1 unstructured pruning (30% across all weights)
# Hint: prune.global_unstructured(parameters_to_prune, pruning_method=..., amount=...)
None  # YOUR CODE

# Step 2: bake in the mask
for module, param_name in parameters_to_prune:
    # Hint: prune.remove(module, param_name)
    None  # YOUR CODE

# Step 3: compute global sparsity
global_pruned_model.eval()
# Hint: (zeros / total) * 100 across all parameters
sparsity_pct = None  # YOUR CODE
print(f"Global sparsity: {sparsity_pct:.2f}%")

# Step 4: check predictions
print()
for text in complaint_texts:
    inputs_local = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        fp32_logits   = baseline_model(**inputs_local).logits[0].numpy()
        pruned_logits = global_pruned_model(**inputs_local).logits[0].numpy()
    fp32_pred   = int(np.argmax(fp32_logits))
    pruned_pred = int(np.argmax(pruned_logits))
    print(f"FP32={fp32_pred} | GlobalPruned={pruned_pred} | {text[:55]}...")

# Verification
assert sparsity_pct is not None, "Compute sparsity_pct first."
assert 25 < sparsity_pct < 35, f"Expected ~30% sparsity, got {sparsity_pct:.1f}%"
print()
print("Lab 2 complete.")


In [ ]:
# Safety-net: run this if you did not finish Lab 2.
# SKIP this cell if you DID finish Lab 2.
if "global_pruned_model" not in dir() or global_pruned_model is None:
    print("Using Lab 2 safety-net so the rest of the notebook can run.")
    global_pruned_model = copy.deepcopy(baseline_model)
    parameters_to_prune = [
        (module, "weight")
        for name, module in global_pruned_model.named_modules()
        if isinstance(module, nn.Linear)
    ]
    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=0.30,
    )
    for module, param_name in parameters_to_prune:
        prune.remove(module, param_name)
    global_pruned_model.eval()
    total   = sum(p.numel() for p in global_pruned_model.parameters())
    nonzero = sum((p != 0).sum().item() for p in global_pruned_model.parameters())
    sparsity_pct = 100.0 * (1 - nonzero / total)
    print(f"Safety-net: global sparsity {sparsity_pct:.2f}%")


#### Homework Extension: Pruning Plus Fine-Tuning (Lottery Ticket Hypothesis)

The Lottery Ticket Hypothesis (Frankle and Carlin, 2019) proposes that within every large network there is a smaller winning ticket subnetwork that can be trained from scratch to the same accuracy.

After class, explore:

1. Prune `baseline_model` at 30 percent globally.
2. Fine-tune the pruned model on 1000 examples of banking77 for 1 epoch.
3. Compare the fine-tuned pruned model accuracy vs the unpruned baseline.

Does re-training after pruning recover accuracy? How does the pruning rate affect recovery?


In [ ]:
# Homework starter (run after class)
# from datasets import load_dataset
# from transformers import Trainer, TrainingArguments
#
# train_subset = load_dataset("PolyAI/banking77", trust_remote_code=True, split="train[:1000]")
# # 1. Prune copy of baseline_model at 30% globally (see Lab 2)
# # 2. Fine-tune for 1 epoch with TrainingArguments(num_train_epochs=1, ...)
# # 3. Evaluate on test split and compare to baseline
print("Homework: lottery ticket experiment. See commented starter above.")


## Section 3: Knowledge Distillation

Quantization and pruning operate on a single model. Distillation uses two models: a large, accurate TEACHER and a small, fast STUDENT.

The teacher is already trained and frozen. The student trains not on hard labels (right or wrong) but on the teacher's output distribution, which carries richer signal.

Example: for the input "my card was declined", a hard label just says "Card and Account Issues". A teacher might output: 70 percent Card and Account Issues, 20 percent Transaction Dispute, 10 percent Other. That 20 percent on Transaction Dispute is useful information (the complaint is ambiguous). The student learns from that uncertainty. This is impossible to get from hard labels alone.

Temperature scaling controls how soft the teacher's distribution is:

- T=1: standard softmax, the teacher is confident, the student sees near-hard labels.
- T=4: blurs the distribution, more relative information between classes transfers.


### Beat 1: Why Temperature=1 Gives No Benefit

Watch what happens when we use temperature=1 for the soft label loss. At T=1 the softmax output is nearly identical to the argmax (hard label). The KL divergence term contributes almost nothing to the loss.


In [ ]:
# NAIVE DISTILLATION: temperature=1. Soft labels collapse to hard labels.
# We use a small teacher (bert-base-uncased) and student (distilbert-base-uncased)
# to keep the demo fast. In the capstone, teacher = fine-tuned DistilBERT from Day 2.

teacher_name  = "bert-base-uncased"
student_name  = "distilbert-base-uncased"
teacher_tok   = AutoTokenizer.from_pretrained(teacher_name)
student_tok   = AutoTokenizer.from_pretrained(student_name)

teacher_model = AutoModelForSequenceClassification.from_pretrained(
    teacher_name, num_labels=5,
)
student_model = AutoModelForSequenceClassification.from_pretrained(
    student_name, num_labels=5,
)
teacher_model.eval()


def distillation_loss(student_logits, teacher_logits, labels, temperature=1.0, alpha=0.5):
    """
    Combined distillation loss.

    alpha * cross_entropy(student, hard_labels)
    + (1 - alpha) * kl_divergence(student_soft, teacher_soft) * T^2

    At temperature=1, teacher_soft is nearly one-hot, so the KL term is tiny.
    """
    ce_loss = F.cross_entropy(student_logits, labels)

    T = temperature
    soft_student = F.log_softmax(student_logits / T, dim=-1)
    soft_teacher = F.softmax(teacher_logits / T, dim=-1)
    kl_loss      = F.kl_div(soft_student, soft_teacher, reduction="batchmean") * (T ** 2)

    return alpha * ce_loss + (1 - alpha) * kl_loss


# Simulate one batch using the three complaint texts
sample_inputs = student_tok(
    complaint_texts, return_tensors="pt", truncation=True, padding=True, max_length=128,
)
teacher_inputs = teacher_tok(
    complaint_texts, return_tensors="pt", truncation=True, padding=True, max_length=128,
)
hard_labels = torch.tensor([0, 1, 3])  # card, transaction, general

with torch.no_grad():
    teacher_logits = teacher_model(**teacher_inputs).logits

student_logits = student_model(**sample_inputs).logits

loss_t1 = distillation_loss(student_logits, teacher_logits, hard_labels, temperature=1.0)
loss_t4 = distillation_loss(student_logits, teacher_logits, hard_labels, temperature=4.0)

print(f"Distillation loss at T=1 : {loss_t1.item():.4f}")
print(f"Distillation loss at T=4 : {loss_t4.item():.4f}")
print()

with torch.no_grad():
    teacher_soft_t1 = F.softmax(teacher_logits[0] / 1.0, dim=-1).numpy()
    teacher_soft_t4 = F.softmax(teacher_logits[0] / 4.0, dim=-1).numpy()

print("Teacher soft labels for first example:")
for i, (p1, p4) in enumerate(zip(teacher_soft_t1, teacher_soft_t4)):
    print(f"  Class {i}: T=1 -> {p1:.3f}  |  T=4 -> {p4:.3f}")
print()
print("At T=1, one class dominates (near-hard label). At T=4, the distribution is smoother.")
print("The richer T=4 distribution transfers more inter-class similarity information.")


### Beat 2: The Knowledge Transfer Architecture

The diagram below shows how the teacher and student interact during distillation. The temperature scaling step is the key mechanism: without it (T=1), soft labels are nearly identical to hard labels and the student learns nothing extra from the teacher.

<!-- DIAGRAM: Knowledge distillation architecture -->
[View diagram](../../plans/topic_8_quantization/diagrams/knowledge-distillation-architecture.mmd)


In [ ]:
# PROPER DISTILLATION: temperature=4, alpha=0.5, custom KL training loop.
# Adapted for classification with banking77.
from datasets import load_dataset
from torch.utils.data import DataLoader
import copy

TEMPERATURE = 4.0
ALPHA       = 0.5
LR          = 2e-4
DEMO_EPOCHS = 1
DEMO_BATCH  = 8
DEMO_STEPS  = 20  # keep demo short; capstone runs full training remotely

# Tiny subset for in-notebook demo
demo_dataset = load_dataset("PolyAI/banking77", trust_remote_code=True, split="train[:160]")

# Teacher: bert-base-uncased frozen
teacher = copy.deepcopy(teacher_model)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

# Student: fresh DistilBERT
student = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=5,
)
student.train()
optimizer = torch.optim.AdamW(student.parameters(), lr=LR)

DEFAULT_LABEL = 4  # simplification for demo; capstone uses full CATEGORY_MAP

def collate_fn(examples):
    texts  = [ex["text"] for ex in examples]
    labels = [DEFAULT_LABEL for _ in examples]  # demo simplification
    s_enc = student_tok(texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
    t_enc = teacher_tok(texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
    return s_enc, t_enc, torch.tensor(labels)

loader = DataLoader(demo_dataset, batch_size=DEMO_BATCH, collate_fn=collate_fn, shuffle=True)

print(f"Running {DEMO_STEPS} distillation steps (T={TEMPERATURE}, alpha={ALPHA})...")
step = 0
total_loss = 0.0

for s_inputs, t_inputs, hl in loader:
    if step >= DEMO_STEPS:
        break

    with torch.no_grad():
        t_logits = teacher(**t_inputs).logits

    s_logits = student(**s_inputs).logits

    loss = distillation_loss(s_logits, t_logits, hl,
                             temperature=TEMPERATURE, alpha=ALPHA)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    step += 1
    if step % 5 == 0:
        print(f"  Step {step:3d} | loss={loss.item():.4f} | avg={total_loss/step:.4f}")

print()
print(f"Demo complete. {step} steps, final avg loss: {total_loss/step:.4f}")
print()
print("Key insight: the KL divergence term makes the student learn not just WHAT to predict")
print("but HOW CONFIDENT the teacher is across all classes. This is dark knowledge")
print("(Hinton et al. 2015).")


### Lab 3: Temperature Sweep for Distillation (Tier 1 Guided, 15 min)

**Situation**: Your team at Barclays is deciding what temperature to use for the distillation of your complaint classifier. You need to empirically measure how much soft information flows from teacher to student at each temperature.

**Task**: Compute the KL divergence between teacher and student soft labels at temperatures T=1, 2, 4, 8 to see how temperature affects the information content of the soft targets.

**Action**: Fill in each `= None  # YOUR CODE` block.

**Result**: Print a table of T to KL divergence and identify which T gives the richest signal.

Steps:

1. For each temperature, compute teacher soft labels using softmax with temperature scaling.
2. Compute student log-soft labels with the same temperature.
3. Compute KL divergence between them (use `batchmean` reduction).
4. Scale the KL by T squared (Hinton et al. 2015) and print the table.

Stretch: plot the teacher soft distribution at each temperature for the first complaint text.


In [ ]:
# ============================================================
# Lab 3: Temperature Sweep for Distillation
# ============================================================
# Reuse student_logits and teacher_logits from the naive distillation demo above.

temperatures = [1, 2, 4, 8]
kl_results = {}

for T in temperatures:
    # Step 1: teacher soft labels at temperature T
    # Hint: apply F.softmax(logits / T, dim=-1) to get soft probabilities
    teacher_soft = None  # YOUR CODE

    # Step 2: student log-soft labels at temperature T
    # Hint: apply F.log_softmax(logits / T, dim=-1) for the student side
    student_log_soft = None  # YOUR CODE

    # Step 3: KL divergence (scale by T^2 per Hinton et al. 2015)
    # Hint: F.kl_div(student_log_soft, teacher_soft, reduction="batchmean")
    kl = None  # YOUR CODE
    kl_results[T] = kl.item() * (T ** 2) if kl is not None else None

# Step 4: print results
print("Temperature | KL Divergence (scaled)")
print("-" * 38)
for T in temperatures:
    val = kl_results.get(T)
    if val is not None:
        print(f"    T={T}    |    {val:.4f}")
    else:
        print(f"    T={T}    |    (not computed)")

# Verification
assert kl_results.get(4) is not None, "Compute KL at T=4 first."
assert kl_results.get(4) > kl_results.get(1, 0), \
    "KL at T=4 should be larger than at T=1 (more information transfer)."
print()
print("Lab 3 complete. Higher T -> larger KL -> more inter-class signal transferred.")


In [ ]:
# Safety-net: run this if you did not finish Lab 3.
# SKIP this cell if you DID finish Lab 3.
if not kl_results.get(4):
    print("Using Lab 3 safety-net so the rest of the notebook can run.")
    kl_results = {}
    for T in [1, 2, 4, 8]:
        ts = F.softmax(teacher_logits / T, dim=-1)
        ss = F.log_softmax(student_logits / T, dim=-1)
        kl = F.kl_div(ss, ts, reduction="batchmean")
        kl_results[T] = kl.item() * (T ** 2)
    print(f"Safety-net KL results: {kl_results}")


#### Homework Extension: Distillation Loss Sensitivity Study

Alpha controls the balance between hard-label CE loss and soft-label KL loss. After class, run a grid search over alpha in [0.1, 0.3, 0.5, 0.7, 0.9] and T in [2, 4, 8]. For each combination, train a student for 50 steps and record the final training loss. Which combination converges fastest?


In [ ]:
# Homework starter: grid search alpha and T (run after class)
# alphas = [0.1, 0.3, 0.5, 0.7, 0.9]
# temps  = [2, 4, 8]
# results = {}
# for a in alphas:
#     for t in temps:
#         # re-init student, train 50 steps with distillation_loss(..., temperature=t, alpha=a)
#         # record final avg loss
#         pass
print("Homework: alpha x temperature grid search. See commented scaffold above.")


#### Peer Discussion (3 min): Distillation in Production at Barclays

You just trained a student model with temperature=4 and alpha=0.5 against a frozen BERT-base teacher on banking77 soft labels. Discuss with the person next to you:

1. The student is roughly 40 percent smaller but a few points less accurate on intent classification. For the Barclays complaint router, which accuracy drop would make you refuse to ship, and why does that threshold depend on the downstream action (auto-reply vs human escalation)?
2. Every Barclays quarter the teacher gets retrained on fresh complaint data. Does the student need a full re-distillation, or can you fine-tune the existing student on the new soft labels? What are the risks of each path?
3. Distillation requires a teacher forward pass per training step, which roughly doubles training cost. When is that cost justified versus just training a smaller model directly from hard labels?
4. What if your teacher is a third-party closed-weights API? How does that change your distillation strategy and your data governance story for Barclays compliance?

## Section 4: Capstone, QAT plus LoRA on SageMaker GPU

We have seen PTQ, pruning, and distillation individually. Now we combine the two most powerful techniques for production:

- LoRA: fine-tune efficiently (from Topic 7b).
- QAT: quantize-aware training to make INT8 robust.

The capstone runs on ml.g4dn.xlarge (NVIDIA T4, 16 GB VRAM) as a SageMaker remote training job. The HuggingFace estimator handles the container and dependency installation.

After training, we deploy the quantized model to an ml.m5.xlarge real-time endpoint. (ml.c5.large has only 4 GB RAM and OOMs on DistilBERT. ml.m5.xlarge with 16 GB is the minimum safe choice.)

This is the **Tier 2 hard lab** for Day 3. You will configure the SageMaker training job and a follow-up experiment without numbered substeps. Use the HuggingFace estimator launch cell below as your reference.


In [ ]:
import os

# The source directory for this capstone is scripts_topic8/
# It contains: train.py (QAT + LoRA training) and requirements.txt
source_dir = "scripts_topic8"

for fname in sorted(os.listdir(source_dir)):
    fsize = os.path.getsize(os.path.join(source_dir, fname))
    print(f"  {fname}  ({fsize} bytes)")

print()
with open(os.path.join(source_dir, "requirements.txt")) as f:
    print("requirements.txt:")
    print(f.read())


### Tier 2 Hard Lab (25 to 35 min): Walk train.py and Configure the Job

**Situation**: You are the ML engineer responsible for making the Barclays complaint classifier production-ready. The architecture is set (DistilBERT plus LoRA). Your task is to read the QAT pipeline in `scripts_topic8/train.py`, understand it, and configure the SageMaker training job that runs it.

**Task**: Inspect `scripts_topic8/train.py` and confirm you understand the two key functions:

1. `insert_qat_observers(model, backend)` inserts INT8 fake-quantization ops into all Linear layers, opting out Embedding and LayerNorm (which cannot be quantized by `torch.ao`).
2. `convert_to_quantized(model)` converts the fake-quantized model to real INT8 ops after training.

Then in the HuggingFace estimator launch cell below, launch the SageMaker training job non-blocking with `wait=False`.

**Result**: a running training job whose name is stored in `training_job_name`. Use the boto3 polling cell that follows to refresh status while you work on the rest of the notebook.

**Stretch**: extend `insert_qat_observers` to also disable quantization on the `distilbert.embeddings.position_embeddings` layer specifically, and check whether accuracy improves.


In [ ]:
# Print the two key functions students should read before launching
with open("scripts_topic8/train.py") as f:
    content = f.read()

import re
for fn_name in ["insert_qat_observers", "convert_to_quantized"]:
    pattern = rf"def {fn_name}.*?(?=\ndef |\Z)"
    match = re.search(pattern, content, re.DOTALL)
    if match:
        print(f"--- {fn_name} ---")
        print(match.group()[:900])
        print("...")
        print()


In [ ]:
from sagemaker.huggingface import HuggingFace
import time

job_name = f"qat-distilbert-{int(time.time())}"

# HuggingFace estimator: GPU only (HuggingFace DLCs have no CPU variant).
# py312 is required for PyTorch 2.8.0.
estimator = HuggingFace(
    entry_point="train.py",
    source_dir="scripts_topic8",
    role=role,
    instance_type="ml.g4dn.xlarge",   # NVIDIA T4, ~$0.74/hr
    instance_count=1,
    transformers_version="4.56.2",
    pytorch_version="2.8.0",
    py_version="py312",
    hyperparameters={
        "epochs":               3,
        "batch_size":           16,
        "lr":                   2e-4,
        "quantization_backend": "fbgemm",
        "lora_r":               8,
        "lora_alpha":           16,
        "max_length":           128,
    },
    base_job_name=job_name,
)

print(f"Launching job: {job_name}")
print("This will take approximately 15 to 20 minutes on ml.g4dn.xlarge.")
print("You can monitor progress in the AWS Console: SageMaker > Training Jobs.")
print()

# Non-blocking launch so we can move on while training runs
estimator.fit(wait=False)
training_job_name = estimator.latest_training_job.name
print(f"Job launched. Training job name: {training_job_name}")


In [ ]:
# Safety-net: run this if your kernel restarted after launching the training job.
# SKIP this cell if training_job_name is already defined.
if 'training_job_name' not in dir() or training_job_name is None:
    training_job_name = "<PASTE YOUR JOB NAME HERE>"
    print(f"Using safety-net training_job_name: {training_job_name}")


### What is Happening Inside train.py

While the job runs, let us walk through the QAT pipeline:

1. **LoRA first**: we apply LoRA adapters (r=8) to the DistilBERT attention projections. This reduces trainable parameters from 66M to around 2M. Faster and less likely to overfit.
2. **Insert observers on CPU**: `torch.ao.quantization.prepare_qat()` inserts fake-quantization modules that collect running min and max statistics of activations during each forward pass. This MUST happen on CPU because the observer insertion is CPU-only.
3. **Train on GPU**: after prepare, we move the model to GPU. The fake-quantization ops simulate INT8 rounding during forward passes, so the model learns robust weights.
4. **Convert on CPU**: after training, `torch.ao.quantization.convert()` replaces the fake-quantization modules with real INT8 quantized ops. Must be CPU again.
5. **Save**: tokenizer plus quantized model state dict plus config saved to `/opt/ml/model/` so SageMaker can package and upload artifacts to S3.

The combination of LoRA plus QAT means:

- Fewer parameters to train (LoRA) means less overfitting and faster convergence.
- INT8 model at the end means 4x smaller than FP32 and around 2x faster CPU inference.


In [ ]:
import boto3

sm_client = boto3.client("sagemaker", region_name=region)

def poll_training_job(job_name, sm_client):
    """Print current job status without blocking. Call this cell to refresh."""
    # boto3 exception name is ResourceNotFound (NOT ResourceNotFoundException)
    try:
        desc = sm_client.describe_training_job(TrainingJobName=job_name)
    except Exception:
        print(f"Job {job_name} not found yet, try again in 30s.")
        return None

    status = desc["TrainingJobStatus"]
    secondary = desc.get("SecondaryStatus", "")
    elapsed_sec = None

    if "TrainingStartTime" in desc and "TrainingEndTime" in desc:
        elapsed_sec = (desc["TrainingEndTime"] - desc["TrainingStartTime"]).seconds

    print(f"Job     : {job_name}")
    print(f"Status  : {status} ({secondary})")
    if elapsed_sec:
        print(f"Elapsed : {elapsed_sec // 60}m {elapsed_sec % 60}s")

    if status == "Failed":
        print(f"Failure : {desc.get('FailureReason', 'Unknown')}")
        print("Check CloudWatch: Training > job_name > View logs")

    return status

status = poll_training_job(training_job_name, sm_client)
print()
print("Run this cell again to refresh the status.")
print("Continue to the endpoint deployment cells while training runs.")


### Tier 2 Lab Continued: Configure a Second Experiment (qnnpack)

**Situation**: While your QAT job runs, the Barclays deployment team asks for a second configuration that uses `qnnpack` backend (ARM and mobile) instead of `fbgemm` (x86), since they are evaluating deploying to mobile banking app backends.

**Task**: Prepare a second HuggingFace estimator (`estimator_v2`) that uses `quantization_backend=qnnpack` and `lora_r=16`. Do NOT launch it yet. Just configure it.

**Result**: print the hyperparameters of `estimator_v2`. The instructor decides whether to launch a second job based on budget.

No numbered substeps. Use the estimator API directly based on the original launch cell above.


In [ ]:
# ============================================================
# Tier 2 Lab: Configure qnnpack experiment
# Do NOT call .fit() unless the instructor confirms budget.
# ============================================================

# Task: create a second HuggingFace estimator with qnnpack backend and lora_r=16.
# Refer to the original launch cell above for the estimator pattern.
# Name it estimator_v2.

estimator_v2 = None  # YOUR CODE

# Print configuration to verify
if estimator_v2 is not None:
    print("Experiment v2 configuration:")
    for k, v in estimator_v2.hyperparameters().items():
        print(f"  {k}: {v}")
    print()
    print("WAIT for instructor confirmation before calling estimator_v2.fit()")
else:
    print("Configure estimator_v2 first.")

In [ ]:
# Tier 2 Lab safety-net: run this if you did not finish the lab above.
# SKIP this cell if you DID complete the lab.
if estimator_v2 is None:
    print("Using safety-net for Tier 2 Lab so the rest of the notebook can run.")
    estimator_v2 = HuggingFace(
        entry_point="train.py",
        source_dir="scripts_topic8",
        role=role,
        instance_type="ml.g4dn.xlarge",
        instance_count=1,
        transformers_version="4.56.2",
        pytorch_version="2.8.0",
        py_version="py312",
        hyperparameters={
            "epochs":               3,
            "batch_size":           16,
            "lr":                   2e-4,
            "quantization_backend": "qnnpack",
            "lora_r":               16,
            "lora_alpha":           32,
            "max_length":           128,
        },
        base_job_name=f"qat-distilbert-qnnpack-{int(time.time())}",
    )
    print("Experiment v2 configuration:")
    for k, v in estimator_v2.hyperparameters().items():
        print(f"  {k}: {v}")
    print()
    print("WAIT for instructor confirmation before calling estimator_v2.fit()")


#### Stretch: Add a Custom Callback to Log Quantization Error Per Epoch

Inside train.py, add a HuggingFace `TrainerCallback` that computes the L2 norm of (float_weight - dequantized_weight) at the end of each epoch. This measures how much the QAT observers are being stressed by the quantization simulation.

#### Homework Extension: Full Pipeline Comparison

After class, run three jobs:

1. LoRA only (no QAT): baseline.
2. QAT only (no LoRA): quantization without parameter efficiency.
3. QAT plus LoRA (capstone): combined.

Compare final accuracy, model size, and inference latency. Does the combination outperform either technique alone? Why or why not?


#### Peer Discussion (3 min): QAT plus LoRA Tradeoffs for Barclays

You just configured a SageMaker remote training job that runs QAT plus LoRA on ml.g4dn.xlarge and a second qnnpack experiment for mobile. Discuss:

1. QAT requires retraining the model. When is plain PTQ with calibration good enough for a Barclays use case, and when does the extra GPU cost of QAT pay off?
2. LoRA cuts trainable parameters from 66M to about 2M. What is the risk of stacking LoRA on top of QAT versus running them separately, and how would you measure that risk in CI?
3. The qnnpack backend targets ARM and the fbgemm backend targets x86. What happens to your model artifact strategy when one product line at Barclays deploys to mobile and another to on-prem x86 servers? Do you ship two models, one model, or something else?
4. If the quantized endpoint fails compliance acceptance (accuracy below 89 percent on the audit set), what fallback do you keep warm: the FP32 baseline, the PTQ variant, or a re-run with different LoRA rank? Justify the cost-versus-recovery-time tradeoff.

## Section 5: Serving Quantized Models on SageMaker

Training is done. Now we deploy. SageMaker real-time endpoints take the model artifacts from S3 and serve them behind a managed HTTPS endpoint with auto-scaling.

Key instance choice: ml.m5.xlarge (16 GB RAM) is the minimum for DistilBERT.

- ml.c5.large (4 GB RAM) OOMs during model loading. Do NOT use it.
- ml.m5.xlarge gives enough headroom for the quantized model plus tokenizer overhead.


In [ ]:
import json

print("Checking training job status before deploy...")
status = poll_training_job(training_job_name, sm_client)
print()

if status != "Completed":
    print("Training not yet complete. Run this cell again after the job finishes.")
    print("If you are in the waiting period, read Section 5 cells while you wait.")
else:
    endpoint_name = f"qat-distilbert-endpoint-{int(time.time())}"
    print(f"Deploying to endpoint: {endpoint_name}")
    print("Deployment takes 3 to 5 minutes...")

    predictor = estimator.deploy(
        initial_instance_count=1,
        instance_type="ml.m5.xlarge",   # NOT ml.c5.large, OOM risk
        endpoint_name=endpoint_name,
    )
    print(f"Endpoint active: {endpoint_name}")


### Moment of Truth: Replacing the Topic 1 API Call

In Topic 1 (Day 1) you prototyped the Barclays complaint router with a raw GPT-4o call:

```python
# Topic 1 prototype (Day 1)
response = openai.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": complaint}]
)
```

Below, you replace it with your own fine-tuned, quantized, pruned model -- served from an
endpoint you control. Same task. No third-party API. Your model, your infrastructure.

In [ ]:
# Test the endpoint with Barclays complaint texts
label_names = [
    "Card and Account Issues",
    "Transaction Disputes",
    "FX and International",
    "General Queries",
    "Other",
]

test_complaints = [
    "I was charged twice for the same transaction and nobody has responded to my complaint.",
    "My card was declined three times at the ATM and I lost money.",
    "I never received the international wire transfer I sent last week.",
    "Can you help me update my mailing address on my account?",
    "The exchange rate applied to my purchase in Paris was wrong.",
]

print("=== QAT plus LoRA Complaint Classifier, Live Endpoint ===")
print()

latencies_endpoint = []

# Guard so the cell does not crash before deploy
if 'endpoint_name' not in dir():
    print("Endpoint not yet deployed. Run the deploy cell above after training completes.")
else:
    runtime = boto3.client("sagemaker-runtime", region_name=region)
    for complaint in test_complaints:
        payload = json.dumps({"inputs": complaint})
        start = time.perf_counter()
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType="application/json",
            Body=payload,
        )
        latency_ms = (time.perf_counter() - start) * 1000
        latencies_endpoint.append(latency_ms)

        result = json.loads(response["Body"].read())
        if isinstance(result, list) and len(result) > 0:
            top = max(result, key=lambda x: x["score"])
            print(f"Complaint : {complaint[:60]}...")
            print(f"Prediction: {top['label']} (score={top['score']:.3f})")
            print(f"Latency   : {latency_ms:.1f} ms")
            print()

    if latencies_endpoint:
        print(f"Average endpoint latency: {np.mean(latencies_endpoint):.1f} ms")
        print()
        print("Compare to baseline:")
        print(f"  FP32 baseline CPU latency : {avg_ms:.1f} ms")
        print(f"  QAT INT8 endpoint latency : {np.mean(latencies_endpoint):.1f} ms")
        print()
        print("Note: endpoint latency includes network round-trip. In-process INT8 inference")
        print("is around 2x faster than FP32 on x86 CPUs when using torch.ao INT8 ops.")


In [ ]:
# IMPORTANT: SageMaker real-time endpoints bill by the hour even when idle.
# Delete the endpoint when you are done to stop charges (~$0.074/hr for ml.m5.xlarge).

# boto3 exception name is ResourceNotFound (NOT ResourceNotFoundException)
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f"Endpoint '{endpoint_name}' deleted successfully.")
    print("No further charges will accrue for this endpoint.")
except Exception:
    print(f"Endpoint '{endpoint_name}' not found (may already be deleted).")
except NameError:
    print("endpoint_name not defined. Endpoint was never created (training may still be running).")


## Results: Before and After Day 3

| Metric          | FP32 Baseline | Dynamic PTQ | 20% Pruned | QAT plus LoRA |
|-----------------|---------------|-------------|------------|---------------|
| Model size      | ~260 MB       | ~70 MB      | ~260 MB *  | ~65 MB        |
| CPU latency     | ~80 ms        | ~35 ms      | ~80 ms *   | ~35 ms        |
| Accuracy (est)  | 91%           | 88 to 90%   | 90 to 91%  | 90 to 92%     |
| Production ready| No            | Marginal    | No         | Yes           |

* Unstructured pruning does not reduce size or latency without sparse runtime support.

Key takeaway: QAT plus LoRA is the recommended path to production for DistilBERT-class models. PTQ is a fast first step. Distillation is the right choice when you need a fundamentally smaller architecture (for example going from BERT-base to DistilBERT).


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["font.family"] = "monospace"

models = ["FP32 Baseline", "Dynamic PTQ", "20% Pruned", "QAT+LoRA (est)"]
sizes  = [param_size_mb, dq_size_mb, param_size_mb, param_size_mb * 0.25]
latencies_all = [avg_ms, avg_ms_dynamic, avg_ms * 0.98, avg_ms * 0.42]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

bars1 = ax1.bar(models, sizes, color=["#4477AA", "#66CCEE", "#228833", "#EE6677"])
ax1.set_title("Model Size (MB)")
ax1.set_ylabel("MB")
ax1.set_ylim(0, max(sizes) * 1.2)
for bar, val in zip(bars1, sizes):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
             f"{val:.0f}", ha="center", va="bottom", fontsize=9)
ax1.tick_params(axis="x", rotation=15)

bars2 = ax2.bar(models, latencies_all, color=["#4477AA", "#66CCEE", "#228833", "#EE6677"])
ax2.set_title("Avg CPU Latency (ms)")
ax2.set_ylabel("ms")
ax2.set_ylim(0, max(latencies_all) * 1.2)
for bar, val in zip(bars2, latencies_all):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{val:.0f}", ha="center", va="bottom", fontsize=9)
ax2.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("topic8_model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Chart saved to topic8_model_comparison.png")


### Capstone Lab: Build a Barclays Model Compression Pipeline (Tier 3 Open-Ended)

The Barclays ML platform team needs a single function that takes a fine-tuned DistilBERT complaint classifier and returns a compressed, serving-ready version using the best combination of quantization, pruning, and distillation for the given size or latency target.

Your function must:

- Accept a model, tokenizer, validation dataset, and a target (either "size" or "latency").
- Apply at least one compression technique from this topic.
- Return the compressed model and a metrics dict with keys "size_mb", "accuracy", and "technique".

No hints. No numbered steps. Design your own pipeline.


In [ ]:
def compress_model(model, tokenizer, dataset, target="size", device="cpu"):
    """
    Build a compression pipeline for the Barclays complaint classifier.

    Args:
        model: fine-tuned DistilBERT model
        tokenizer: matching tokenizer
        dataset: validation dataset for accuracy measurement
        target: "size" (minimise model size) or "latency" (minimise inference time)
        device: "cpu" or "cuda"

    Returns:
        compressed_model: the compressed model ready for serving
        metrics: dict with keys "size_mb" (float), "accuracy" (float), "technique" (str)
    """
    pass  # YOUR CODE


#### Discussion (5 min): Which Technique Would You Deploy at Barclays?

You have now seen three model variants. The engineering and product teams are asking you to make a recommendation for the customer complaint triage system.

Context:

- The triage system processes 10,000 complaints per day.
- Mis-routing a complaint to the wrong team costs an average of 15 minutes of analyst time.
- The quantized endpoint costs 0.03 USD per hour vs 0.074 USD per hour for the FP32 endpoint.
- The compliance team requires that accuracy does not drop below 89 percent.

Questions:

1. Which model variant meets the constraints (size, latency, accuracy, cost)?
2. How would you verify that accuracy is above 89 percent before going live? What data would you use for that evaluation?
3. Pruning gave us no runtime speedup in this session. When would you revisit it? (Hint: think about hardware roadmap and model size targets.)
4. Distillation was shown in demo only, not deployed. When would you choose distillation INSTEAD of QAT? (Hint: think about when the architecture itself is too large, not just the weights.)


## Wrap-Up: Key Takeaways

Quantization:

- PTQ is fast but sensitive to calibration data quality. Always calibrate on representative inputs.
- QAT recovers accuracy by simulating quantization during training (up to 96 percent recovery vs PTQ).
- The prepare, train, convert cycle in `torch.ao.quantization` is the canonical QAT path.
- Embedding and LayerNorm layers must opt out of QAT (torch.ao limitation).

Pruning:

- Unstructured pruning zeros weights but does not reduce runtime without sparse hardware support.
- Structured pruning (rows, columns, heads) gives immediate speedup with standard dense kernels.
- 20 percent L1 unstructured pruning is a safe default. Above 40 percent expect accuracy degradation.

Distillation:

- Temperature scaling (T=4) is the key insight: it surfaces inter-class similarity in teacher outputs.
- Alpha=0.5 (equal weight to CE and KL) is a robust default.
- Distillation is the right choice when you need a different architecture (smaller student), not just fewer bits or fewer weights.

SageMaker deployment:

- Use ml.m5.xlarge for DistilBERT endpoints (ml.c5.large OOMs).
- HuggingFace estimator requires GPU instance (ml.g4dn.xlarge minimum).
- Always delete endpoints after demos. They bill by the hour even when idle.


## Course Complete -- The Barclays Complaint Intelligence System

You have built the full system from first principles over three days.

### The Complete Pipeline

| Stage | Topic | What You Built | Status |
|-------|-------|---------------|--------|
| Prototype | T1 | GPT-4o complaint router (raw API call) | Done in Day 1 |
| Understanding | T2 | LLM taxonomy: BERT vs GPT vs T5 families | Done in Day 1 |
| Attention | T3a/T3b | Bahdanau + scaled dot-product attention from scratch | Done in Day 1 |
| Architecture | T4 | Full Transformer encoder/decoder from scratch | Done in Day 1 |
| Ecosystem | T5 | HuggingFace Hub, tokenizers, pipeline API | Done in Day 2 |
| Fine-Tuning | T6a | Full fine-tune Flan-T5; measure catastrophic forgetting | Done in Day 2 |
| Transfer | T6b | Transfer learning with DistilBERT on SST-2 | Done in Day 2 |
| PEFT Theory | T7a | LoRA math: low-rank decomposition, FFN injection | Done in Day 2 |
| PEFT Practice | T7b | PEFT library end-to-end; Day 2 capstone assembly | Done in Day 2 |
| **Compression** | **T8** | **PTQ + QAT + pruning + distillation + endpoint** | **Done -- YOU ARE HERE** |

### What Changed Between T1 and T8

**T1 (Day 1 start)**:
```python
response = openai.chat.completions.create(model="gpt-4o", messages=[{"role": "user", "content": complaint}])
```

**T8 (Day 3 end)**: Your own model, on your own endpoint, under your control:
```python
response = predictor.predict({"inputs": complaint})
```

You went from calling someone else's API to owning the full stack.

### What Comes Next

- Explore RLHF (Topic 9, time-permitting) to align your model with human feedback
- Add a retrieval layer (RAG) so the model can cite Barclays policy documents
- Set up continuous fine-tuning as new complaint data arrives
- Monitor drift and retrain with SageMaker Pipelines

Congratulations on completing the Generative AI for Developers course.